# 서울 공공자전거(따릉이) 신규가입자 & 외국인 대여 데이터 분석

이 노트북은 서울시 공공자전거의 두 공개 데이터셋을 정제·통합하여 인사이트를 도출합니다.

- **서울특별시 공공자전거 신규가입자 정보(일별)**: 2017-01 ~ 2021-06
- **서울특별시 공공자전거 외국인 대여정보(월별)**: 2017-01 ~ 2021-06

> 기존 노트북(`Seoul_bike_new_subscribers.ipynb`, `monthly-foreigner-rental-of-public-bicycles-in-Seoul.ipynb`)은 로컬 Windows 경로(`C:/Users/...`)에 하드코딩되어 있어 다른 환경에서 실행할 수 없었습니다. 이 노트북은 저장소 상대 경로를 사용해 어디서나 재현할 수 있도록 다시 작성했고, 두 데이터셋을 한 번에 정제·비교하며 실행 가능한 개선 액션까지 정리합니다.

**요약 결론**: 국내 신규가입은 코로나 기간 오히려 급증(+35.5% YoY)한 반면, 외국인 대여는 급감(-62.9% YoY)한 뒤 2021년 상반기까지도 회복되지 못했습니다. 이용은 주말·2030세대·소수 관광 거점에 강하게 쏠려 있습니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from pathlib import Path

plt.rcParams['axes.unicode_minus'] = False
# 한글 폰트가 설치되어 있다면 다음 줄의 주석을 해제하세요.
# plt.rc('font', family='NanumGothic')  # Linux/Windows
# plt.rc('font', family='AppleGothic')  # macOS

BASE = Path('.')
D1 = BASE / '서울특별시 공공자전거 신규가입자 정보(일별)'
D2 = BASE / '서울특별시 공공자전거 외국인 대여정보(월별)'

## 1. 데이터 로드 및 정제

두 데이터셋 모두 파일마다 인코딩(CP949/UTF-8), 컬럼명, 확장자(일부 `.xlsx`가 `.csv`로 잘못 저장됨)가 제각각이라 통일된 로더가 필요합니다.

In [ ]:
AGE_MAP = {
    "'~10대'": 'AGE_001', "'20대'": 'AGE_002', "'30대'": 'AGE_003', "'40대'": 'AGE_004',
    "'50대'": 'AGE_005', "'60대'": 'AGE_006', "'70대~'": 'AGE_007',
    '~10대': 'AGE_001', '20대': 'AGE_002', '30대': 'AGE_003', '40대': 'AGE_004',
    '50대': 'AGE_005', '60대': 'AGE_006', '70대~': 'AGE_007',
}
AGE_LABEL = {
    'AGE_001': '~10대', 'AGE_002': '20대', 'AGE_003': '30대', 'AGE_004': '40대',
    'AGE_005': '50대', 'AGE_006': '60대', 'AGE_007': '70대~', 'AGE_008': '기타/외국인/비회원',
}

subs_frames = []

def add_subs(df, source):
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    df = df.rename(columns={'가입 수': '신규가입자수'})
    keep = ['가입일자', '사용자코드', '연령대코드', '성별', '신규가입자수']
    for c in keep:
        if c not in df.columns:
            df[c] = np.nan
    df = df[keep]
    for c in ['가입일자', '사용자코드', '연령대코드', '성별']:
        if pd.api.types.is_string_dtype(df[c]) or df[c].dtype == object:
            df[c] = df[c].astype('string').str.strip().str.strip("'").replace({'nan': pd.NA})
    df['연령대코드'] = df['연령대코드'].replace(AGE_MAP)
    df['가입일자'] = pd.to_datetime(df['가입일자'], errors='coerce')
    df['신규가입자수'] = pd.to_numeric(df['신규가입자수'], errors='coerce')
    df['source'] = source
    subs_frames.append(df)

# 2017, 2018 상반기: CP949 CSV (컬럼값이 '2017-01-01' 처럼 따옴표로 감싸져 있음)
df = pd.read_csv(D1 / "서울특별시 공공자전거 신규가입자 정보(일별)_2017년.csv", encoding='cp949')
df.columns = ['가입일자', '사용자코드', '성별', '연령대코드', '신규가입자수']
add_subs(df, '2017')

df = pd.read_csv(D1 / "서울특별시 공공자전거 신규가입자 정보(일별)_2018년.csv", encoding='cp949')
df.columns = ['가입일자', '사용자코드', '성별', '연령대코드', '신규가입자수']
add_subs(df, '2018_H1')  # 2018-01-01 ~ 2018-07-19

# 2018 하반기, 2019: xlsx
add_subs(pd.read_excel(D1 / "서울특별시 공공자전거 신규가입자 정보(일별)_20180720_20181231.xlsx"), '2018_H2')
add_subs(pd.read_excel(D1 / "서울특별시 공공자전거 신규가입자 정보(일별)_20190101_20191130.xlsx"), '2019')

# 2019.12~2020.06: 확장자만 .csv이고 실제로는 xlsx (Excel 2007+) 파일
add_subs(pd.read_excel(D1 / "서울특별시 공공자전거 신규가입자 정보(일별)_2019.12.01_2020.05.31.csv"), '2019.12-2020.05')
add_subs(pd.read_excel(D1 / "서울특별시 공공자전거 신규가입자 정보(일별)_2020.06.01_2020.06.30.csv"), '2020.06')

# 2020.07 ~ 2021.06: 정상 CP949 CSV
for fname, src in [
    ("서울특별시 공공자전거 신규가입자 정보(일별)_2020.07.01_2020.08.31.csv", '2020.07-08'),
    ("서울특별시 공공자전거 신규가입자 정보(일별)_2020.09.01_2020.09.30.csv", '2020.09'),
    ("서울특별시 공공자전거 신규가입자 정보(일별)_2020.10.01_2020.10.31.csv", '2020.10'),
    ("서울특별시 공공자전거 신규가입자 정보(일별)_2020.11.01_2020.11.30.csv", '2020.11'),
    ("서울특별시 공공자전거 신규가입자 정보(일별)_2020.12.01_2020.12.31.csv", '2020.12'),
    ("서울특별시 공공자전거 신규가입자 정보(일별)_2021.01.01_2021.01.31.csv", '2021.01'),
    ("서울특별시 공공자전거 신규가입자 정보(일별)_21.02-21.06.csv", '2021.02-06'),
]:
    add_subs(pd.read_csv(D1 / fname, encoding='cp949'), src)

subs = pd.concat(subs_frames, ignore_index=True).dropna(subset=['가입일자', '신규가입자수'])
subs['연령대라벨'] = subs['연령대코드'].map(AGE_LABEL)
subs['year'] = subs['가입일자'].dt.year
subs['month'] = subs['가입일자'].dt.month
subs['weekday'] = subs['가입일자'].dt.weekday  # 0=월 ... 6=일
subs = subs.sort_values('가입일자').reset_index(drop=True)

print(f"신규가입자 데이터: {len(subs):,}행, {subs['가입일자'].min().date()} ~ {subs['가입일자'].max().date()}")
subs.head()

In [ ]:
rent_frames = []

def add_rent(df, source):
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    df = df.rename(columns={'일시': '대여일자'})
    keep = ['대여일자', '대여소번호', '대여소', '대여건수', '반납건수']
    for c in keep:
        if c not in df.columns:
            df[c] = np.nan
    df = df[keep]
    for c in ['대여일자', '대여소번호', '대여소']:
        if pd.api.types.is_string_dtype(df[c]) or df[c].dtype == object:
            df[c] = df[c].astype('string').str.strip().str.strip("'").replace({'nan': pd.NA})
    df['대여일자'] = df['대여일자'].astype(str).str.strip("'").str[:6]  # YYYYMM
    df['대여건수'] = pd.to_numeric(df['대여건수'], errors='coerce')
    df['반납건수'] = pd.to_numeric(df['반납건수'], errors='coerce')
    df['source'] = source
    rent_frames.append(df)

df = pd.read_csv(D2 / "서울특별시 공공자전거 외국인 대여정보(월별)_2017년.csv", encoding='cp949')
df.columns = ['대여일자', '대여소번호', '대여소', '대여건수', '반납건수']
add_rent(df, '2017')

df = pd.read_csv(D2 / "서울특별시 공공자전거 외국인 대여정보(월별)_2018년.csv", encoding='cp949')
df.columns = ['대여일자', '대여소번호', '대여소', '대여건수', '반납건수']
add_rent(df, '2018')  # 원자료에 1~6월만 존재 (하반기 누락)

add_rent(pd.read_excel(D2 / "서울특별시 공공자전거 외국인 대여정보(월별)_201901_201911.xlsx"), '2019.01-11')

tmp = pd.read_excel(D2 / "서울특별시 공공자전거 외국인 대여정보(월별)_201912_202005.xlsx")
add_rent(tmp, '2019.12-2020.05')
rent_frames[-1] = rent_frames[-1][rent_frames[-1]['대여일자'].astype(int) >= 201912]

add_rent(pd.read_excel(D2 / "서울특별시 공공자전거 외국인 대여정보(월별)_2020.06.xlsx"), '2020.06')
add_rent(pd.read_excel(D2 / "서울특별시 공공자전거 외국인 대여정보(월별)_2020.07_2021.01.xlsx"), '2020.07-2021.01')
add_rent(pd.read_csv(D2 / "서울특별시 공공자전거 외국인 대여정보(월별)_2021.02_2021.06.csv", encoding='cp949'), '2021.02-06')

rent = pd.concat(rent_frames, ignore_index=True).dropna(subset=['대여일자'])
rent['대여일자'] = rent['대여일자'].astype(str)
rent = rent[rent['대여일자'].str.match(r'^\d{6}$')].drop_duplicates()
rent['year'] = rent['대여일자'].str[:4].astype(int)
rent['month'] = rent['대여일자'].str[4:6].astype(int)
# 대여소 번호 표기 유무가 혼재되어("207. 여의나루역 1번출구 앞" vs " 여의나루역 1번출구 앞") 있어 정규화
rent['대여소_norm'] = rent['대여소'].astype(str).str.strip().str.replace(r'^\d+(-\d+)?\.\s*', '', regex=True).str.strip()

print(f"외국인 대여 데이터: {len(rent):,}행, {rent['대여일자'].min()} ~ {rent['대여일자'].max()}")
print(f"대여소: 정규화 전 {rent['대여소'].nunique():,}개 -> 정규화 후 {rent['대여소_norm'].nunique():,}개")
rent.head()

### 데이터 커버리지 확인

두 데이터셋 모두 매년 12개월이 온전히 존재하는 것은 아닙니다. 연도 간 비교 시 이 점을 반드시 고려해야 합니다.

In [ ]:
print("신규가입자 - 연도별 존재 월:")
for y, g in subs.groupby('year'):
    print(f"  {y}: {sorted(g['month'].unique())}")

print("\n외국인 대여 - 연도별 존재 월:")
for y, g in rent.groupby('year'):
    print(f"  {y}: {sorted(g['month'].unique())}")
# -> 신규가입자 2021년은 1~6월만, 외국인 대여는 2018년 1~6월만·2021년 1~6월만 존재
# 연도 합계 대신 '월평균'으로 비교해야 공정합니다.

## 2. 신규가입자 인사이트

### 2.1 연도별 성장 추이 (월평균 기준)

In [ ]:
subs_monthly = subs.groupby(['year', 'month'])['신규가입자수'].sum().reset_index()
subs_year_avg = subs_monthly.groupby('year')['신규가입자수'].mean()
print(subs_year_avg.round(0))

fig, ax = plt.subplots(figsize=(7, 4))
subs_year_avg.plot.bar(ax=ax, color='#12897B')
ax.set_title('연도별 월평균 신규가입자수')
ax.set_ylabel('월평균 신규가입자수')
plt.tight_layout()
plt.show()

yoy_2020 = (subs_year_avg[2020] / subs_year_avg[2019] - 1) * 100
print(f"\n2020년 YoY(월평균, vs 2019): {yoy_2020:+.1f}%  ->  코로나 기간 대중교통 회피 수요로 신규가입이 오히려 급증")

**인사이트**: 신규가입자는 2017년(월평균 32,217명)부터 2020년(91,029명)까지 매년 꾸준히 증가했습니다. 특히 2020년은 코로나로 대중교통 이용을 꺼리는 수요가 몰리며 전년 대비 **+35.5%** 급증했습니다.

### 2.2 계절성 & 요일 패턴

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

full_years = subs[subs['year'].isin([2017, 2018, 2019, 2020])]
season = full_years.groupby('month')['신규가입자수'].sum()
season.plot.bar(ax=axes[0], color='#12897B')
axes[0].set_title('월별(계절) 신규가입 총량 - 2017~2020 합산')
axes[0].set_xlabel('월')

weekday = subs.groupby('weekday')['신규가입자수'].sum()
weekday.index = ['월', '화', '수', '목', '금', '토', '일']
colors = ['#A9DDC9'] * 5 + ['#12897B'] * 2
weekday.plot.bar(ax=axes[1], color=colors)
axes[1].set_title('요일별 신규가입 총량 - 전체 기간')

plt.tight_layout()
plt.show()

weekday_avg = subs.groupby('weekday')['신규가입자수'].sum()
weekday_ratio = weekday_avg[[5, 6]].mean() / weekday_avg[[0, 1, 2, 3, 4]].mean()
print(f"주말/평일 비율: {weekday_ratio:.2f}배")

**인사이트**: 3~6월과 9~10월(봄·가을)에 신규가입이 집중되고 1·11·12월(겨울)에 최저치를 기록합니다. 요일별로는 토·일요일이 평일 평균보다 약 1.6배 높아, 통근보다 **레저 목적 가입 비중이 크다**는 점을 시사합니다.

### 2.3 연령대 분포

⚠️ **데이터 품질 이슈**: 2018~2019년 일부 원자료는 "기타/외국인/비회원(AGE_008)" 이용자가 별도 코드 없이 "70대 이상(AGE_007)"에 합산되어 있습니다(2018년 상반기 70대 이상 75,857명은 60대의 33배로 비현실적인 수치). 이 문제가 없는 **2017·2020·2021년**만 사용해 연령 분석을 수행합니다.

In [ ]:
clean_years = [2017, 2020, 2021]
sub_clean = subs[subs['year'].isin(clean_years) & (subs['연령대코드'] != 'AGE_008')]

age_labels = ['~10대', '20대', '30대', '40대', '50대', '60대', '70대~']
age_dist = sub_clean.groupby('연령대라벨')['신규가입자수'].sum().reindex(age_labels)

fig, ax = plt.subplots(figsize=(7, 4))
age_dist.plot.bar(ax=ax, color=['#DCF3E9','#78CBA8','#4FB88C','#2F9E74','#1E8560','#146B4C','#0B4F39'])
ax.set_title('연령대별 신규가입자 (2017·2020·2021 합산)')
plt.tight_layout()
plt.show()

print(f"20대 비중: {age_dist['20대'] / age_dist.sum() * 100:.1f}%")

age_year = sub_clean.groupby(['year', '연령대라벨'])['신규가입자수'].sum().unstack()
age_year_pct = (age_year.div(age_year.sum(axis=1), axis=0) * 100).round(1)
age_year_pct[age_labels]

**인사이트**: 20대가 전체 신규가입자의 약 47%로 압도적 1위입니다. 다만 연도별 추이를 보면 20대 비중은 2017년 53.5% → 2021년 41.5%로 **축소**되는 반면, 40대(12.2%→14.7%)·50대(4.7%→7.1%)·60대(1.0%→2.1%) 비중은 **꾸준히 확대**되고 있어 이용자층이 서서히 다변화되고 있습니다.

## 3. 외국인 대여 인사이트

### 3.1 연도별 추이 & 코로나 영향

In [ ]:
rent_monthly = rent.groupby(['year', 'month'])['대여건수'].sum().reset_index()
rent_year_avg = rent_monthly.groupby('year')['대여건수'].mean()
print(rent_year_avg.round(0))

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#F3C6AE', '#F3C6AE', '#A03D14', '#B3271F', '#F3C6AE']
rent_year_avg.plot.bar(ax=ax, color=colors)
ax.set_title('연도별 월평균 외국인 대여건수\n(2018·2021은 반년치 데이터)')
plt.tight_layout()
plt.show()

yoy_2020_rent = (rent_year_avg[2020] / rent_year_avg[2019] - 1) * 100
recovery_2021 = rent_year_avg[2021] / rent_year_avg[2019] * 100
print(f"\n2020년 YoY(월평균, vs 2019): {yoy_2020_rent:+.1f}%")
print(f"2021년 상반기 수준(2019년 대비): {recovery_2021:.0f}%  ->  아직 코로나 이전 절반 이하")

**인사이트**: 외국인 대여는 2017년(월평균 1,174건) → 2019년(2,231건)까지 약 2배 성장했지만, 2020년 코로나로 **-62.9%** 급감했습니다. 2021년 상반기(990건)에도 2019년의 44% 수준에 그쳐, 국내 신규가입 회복(오히려 증가)과 뚜렷하게 대비됩니다.

### 3.2 계절성

In [ ]:
rent_full = rent[rent['year'].isin([2017, 2019, 2020])]  # 온전한 연도만 사용
rent_season = rent_full.groupby('month')['대여건수'].sum()

fig, ax = plt.subplots(figsize=(7, 4))
top3 = set(rent_season.sort_values(ascending=False).index[:3])
colors = ['#A03D14' if m in top3 else '#F3C6AE' for m in rent_season.index]
rent_season.plot.bar(ax=ax, color=colors)
ax.set_title('월별(계절) 외국인 대여 총량 - 2017·2019·2020 합산')
plt.tight_layout()
plt.show()

**인사이트**: 봄(4~5월)과 가을(9~10월) 관광 성수기에 대여가 집중됩니다 — 국내 신규가입 계절성과 유사한 패턴입니다.

### 3.3 대여소 쏠림도 (Top 10 & 집중도)

In [ ]:
station_sum = rent.groupby('대여소_norm')['대여건수'].sum().sort_values(ascending=False)
top10 = station_sum.head(10)

fig, ax = plt.subplots(figsize=(8, 5))
top10.iloc[::-1].plot.barh(ax=ax, color='#12897B')
ax.set_title('외국인 대여 Top 10 대여소 (누적)')
plt.tight_layout()
plt.show()

n = len(station_sum)
total = station_sum.sum()
for pct in [0.05, 0.1, 0.2]:
    k = max(1, int(n * pct))
    share = station_sum.iloc[:k].sum() / total * 100
    print(f"상위 {pct*100:.0f}% 대여소({k}개)가 전체 대여량의 {share:.1f}% 차지")
print(f"전체 대여소 수: {n:,}개")

**인사이트**: 여의나루역(한강공원)이 2,361건으로 2위 홍대입구역(1,015건)의 2배 이상인 압도적 1위입니다. 상위 5% 대여소(94곳)가 전체 대여량의 47.4%, 상위 20%가 82.4%를 차지할 만큼 한강공원·홍대·서울숲·잠실 등 **소수 관광 거점에 극심하게 쏠려** 있습니다.

## 4. 개선 액션 플랜

세 가지 핵심 인사이트에서 바로 이어지는 실행 항목을 시급도 기준으로 정리했습니다.

### 즉시 실행 (0-3개월)
1. **외국인 대여 회복 캠페인** — 여의나루·홍대입구·서울숲·롯데월드타워 등 상위 8개 대여소에 다국어(영/중/일) QR 안내와 관광공사·호텔 제휴 프로모션을 재가동합니다. *(근거: 2021년 상반기 대여량이 2019년의 44% 수준)*
2. **대여소 재배치 우선순위 조정** — 반납 초과 대여소(고속터미널역, 봉은사역 등)와 대여 초과 대여소(여의도역, 광화문 등)를 연결하는 순환 배치를 우선 적용합니다.
3. **주말 가동률 점검** — 한강공원·홍대 등 주말 성수 지역의 거치대 포화 여부를 우선 점검합니다. *(근거: 주말 신규가입이 평일 평균 대비 +61%)*

### 중기 과제 (3-12개월)
1. **평일 통근 수요 개발** — 정기권 통근 할인, 지하철 환승 연계 프로모션으로 평일 이용률을 끌어올립니다.
2. **고령층 온보딩 개선** — 전기자전거 체험존, 큰 글씨 앱 UI, 오프라인 가입 창구 확대. *(근거: 60대 이상 비중 2.1%로 전 연령대 중 최저)*
3. **성수기 대비 정비 인력 선배치** — 피크 직전인 2월·8월에 정비 인력을 집중 배치해 성수기 초입 결품을 방지합니다.

### 장기 과제 (1년 이상)
1. **인바운드 채널 다변화** — 특정 채널 의존도를 낮춰 향후 충격에 대한 완충력을 확보합니다. *(근거: 2020년 외국인 대여 -62.9% 급락)*
2. **중장년층 생애주기 마케팅** — 40·50·60대 비중이 매년 확대되는 추세를 살려 건강·웰니스 테마 마케팅을 강화합니다.
3. **대여소 쏠림 완화** — 나머지 대여소 인근 로컬 코스·골목상권 콘텐츠를 관광 상품과 연계해 분산을 유도합니다.

## 5. 데이터 한계 및 방법론 노트

- 신규가입자 데이터는 2021년 1~6월까지만 존재해 연도 간 비교는 총합이 아닌 **월평균** 기준으로 계산했습니다.
- 외국인 대여 데이터는 2018년 1~6월 데이터만 공공데이터에 존재하며 하반기(7~12월) 파일이 누락되어 있고, 2021년도 6월까지만 존재합니다.
- 파일마다 인코딩(CP949/UTF-8)이 다르고, 일부 `.xlsx` 파일이 `.csv` 확장자로 잘못 저장되어 있어 원본을 그대로 병합할 수 없었습니다.
- 대여소명에 대여소 번호가 붙는 표기와 빠지는 표기가 혼재되어 있어 번호를 제거해 정규화한 뒤 집계했습니다(정규화 전 2,732개 → 정규화 후 1,889개 대여소).
- 2018~2019년 일부 원자료에서 "기타/외국인/비회원" 이용자가 "70대 이상" 연령 코드에 합산되어 있는 것으로 확인되어, 해당 두 해는 연령대 분석에서 제외하고 2017·2020·2021년만 사용했습니다.
- 본 분석은 서술 통계 기반이며, 제시된 관계는 상관관계로 인과관계를 증명하지 않습니다.